### Ready to Go? Let's Start Your Chemical Discovery with SMILES-Psychoactive Substances-Classifiers!

💡 **Pro Tip:** Want to speed things up? You can use a powerful T4 GPU for free!

Just click on **Runtime** at the top of the page, select **Change runtime type**, and choose **T4 GPU** from the *Hardware accelerator* dropdown.

If a GPU is not available, you can proceed using a CPU runtime. However, please keep the following in mind:
*   All tasks will take longer to complete.


<hr style="border: none; border-top: 2px dashed #ccc;">

#### 🚀 Final Pre-Flight Check (Crucial!)

1.  **Check Your Connection Status** at the top-right of the page.
    - Colab often connects automatically to your last-used device. If you don't see `Connecting...`, please click the **`Connect`** button manually.
2.  **Wait for the Green Checkmark** (✓) to appear next to your `RAM` and `Disk` status. This confirms you are **"Connected"**.
3.  When Colab asks for permission to access your files, go ahead and click **"Connect to Google Drive"** to authorize it.
4.  Once connected, Run Cells **"Step-by-Step"**.

<hr style="border: none; border-top: 2px dashed #ccc;">

#### ✨ Just provide a chemical SMILES, and the PS-SMILES-Classifier will instantly give you its classification. After running, get ready to see :

* **Batch test results on our test set**
* **NPS test results**
* **Batch test results on our test set**

---

### **Step 1: Project Setup & Data Download** ⚙️

<div style="background-color: #e6f7ff; border-left: 6px solid #1890ff; padding: 15px; margin: 15px 0; border-radius: 5px;">
    <h4 style="color: #0050b3; margin-top: 0; font-size: 1.1em;">
    </h4>
    <p>
        This first code cell is the starting point for your entire discovery journey. It will automatically handle all the essential preparations for you.
    </p>
    <p>
        <strong>What it does:</strong>
    </p>
    <ul style="margin-top: 0; padding-left: 20px;">
        <li>Connects to your Google Drive</strong> to save and access data.</li>
        <li>Clones or updates our GitHub repository</strong> to ensure you're using the latest code.</li>
        <li>Installs all required Python libraries</strong> to power the analysis.</li>
        <li>Downloads all model weights and datasets</strong> from Hugging Face.</li>
    </ul>
    <p style="font-weight: bold;">
        To begin, simply run the code cell directly below this one.
    </p>
</div>

In [ ]:
#@title 🧠 1: Setting Up Your Workspace
# ==============================================================================
#  CSU-IR Project Setup Script for Colab
# ==============================================================================
import os
import shutil
from google.colab import drive
from google.colab import userdata
from huggingface_hub import hf_hub_download # Import at the top

# ==================================================================
# --- 1. Mount Google Drive ---
# ==================================================================
print("📁 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True) # Use force_remount for robustness
print("✅ Google Drive mounted.")


# ==================================================================
# --- 2. Define All Project Paths ---
# ==================================================================
print("\n--- Defining Project Paths ---")
# Project Root
GDRIVE_REPO_PATH = "/content/drive/MyDrive/colab_repos/CSU-IR"

# Requirements file
REQUIREMENTS_FILE = os.path.join(GDRIVE_REPO_PATH, 'requirements_colab.txt')

# --- Data and Model Destination Folders on Google Drive ---
# These are the folders where downloaded files will be saved.
DEST_PS_CLASSIFIER_WEIGHTS_PATH = os.path.join(GDRIVE_REPO_PATH, 'PS-Classifier', 'check_points')
print("✅ Paths defined.")


# ==================================================================
# --- 3. Clone or Update the 'CSU-IR' repository ---
# ==================================================================
print("\n--- Cloning or Updating Repository ---")
try:
    GIT_TOKEN = userdata.get('GITHUB_PAT')
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your GitHub Personal Access Token in Colab Secrets (secret name: GITHUB_PAT)")

if not os.path.exists(GDRIVE_REPO_PATH):
    print(f"⏳ Cloning 'CSU-IR' to: {GDRIVE_REPO_PATH}")
    !git clone -q https://{GIT_TOKEN}@github.com/Hsqcsu/CSU-IR.git {GDRIVE_REPO_PATH}
else:
    print(f"✅ Repository already exists. Pulling latest changes...")
    !cd {GDRIVE_REPO_PATH} && git pull


# ==================================================================
# --- 4. Install Dependencies ---
# ==================================================================
print("\n--- Installing Dependencies ---")
if os.path.exists(REQUIREMENTS_FILE):
    print("⏳ Installing dependencies from requirements_colab.txt...")
    !pip install -q -r {REQUIREMENTS_FILE}
    print("✅ Dependencies installed.")
else:
    print(f"⚠️ Warning: Could not find requirements file at {REQUIREMENTS_FILE}.")


# ==================================================================
# --- 5. Define Download Helper and File Lists ---
# ==================================================================
print("\n--- Preparing for Download ---")
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your Hugging Face Token in Colab Secrets (secret name: HF_TOKEN)")

# --- Reusable Download Helper Function ---
def download_files_from_hf(repo_id, files_dict, destination_dir, token):
    """
    Downloads a dictionary of files from a Hugging Face repo to a destination directory.
    """
    print(f"\n--- Checking files for: {os.path.basename(destination_dir)} ---")
    os.makedirs(destination_dir, exist_ok=True)

    for hf_path, local_name in files_dict.items():
        target_path = os.path.join(destination_dir, local_name)
        if not os.path.exists(target_path):
            print(f"  -> Downloading '{local_name}'...")
            try:
                downloaded_path = hf_hub_download(repo_id=repo_id, filename=hf_path, token=token)
                shutil.copy(downloaded_path, target_path)
                print(f"  ✅ Downloaded successfully.")
            except Exception as e:
                print(f"  ❌ FAILED to download '{local_name}'. Error: {e}")
        else:
            print(f"  👍 '{local_name}' already exists.")

# --- Define All Files to be Downloaded ---
HF_REPO_ID = "Skylight666/CSU-IR"
PS_Classifier_MODEL_WEIGHTS_TO_DOWNLOAD = {
    "Model_weights/PS_Classifier/best_ir_model_650-4000.pth": "best_ir_model_650-4000.pth",
    "Model_weights/PS_Classifier/best_smiles_model_650-4000.pth": "best_smiles_model_650-4000.pth",
    "Model_weights/PS_Classifier/best_IR_Classifier_model.pth": "best_IR_Classifier_model.pth",
    "Model_weights/PS_Classifier/best_Smiles_Classifier_model.pth": "best_Smiles_Classifier_model.pth",
}
print("✅ Download helper and file lists are ready.")

# ==================================================================
# --- 6. Execute All Downloads ---
# ==================================================================
print("\n--- Executing All Downloads ---")
download_files_from_hf(HF_REPO_ID, PS_Classifier_MODEL_WEIGHTS_TO_DOWNLOAD, DEST_PS_CLASSIFIER_WEIGHTS_PATH, HF_TOKEN)

# ==================================================================
# --- Finalization ---
# ==================================================================
print("\n\n🎉🎉🎉 Full project setup is complete! 🎉🎉🎉")
print('--------------------------------------------------------------------------------------')
'''
print("\nInstallation complete. The runtime will now restart automatically.")
print("Please wait for the session to reconnect and then run the next cell.")

# This command will automatically restart the Colab runtime.
import os
os.kill(os.getpid(), 9)
print("After re-connected, you can run the analysis and training notebooks.")
'''

---
### **Step 2: Model Initialization & Data Loading**
<div style="background-color: #e6f7ff; border-left: 6px solid #1890ff; padding: 15px; margin: 15px 0; border-radius: 5px;">
    <h4 style="color: #0050b3; margin-top: 0;">
        💡 A Note on Environment Initialization
    </h4>
    <p>
        After you run this cell, you might see some error messages appear briefly in the output. <strong>Please don't worry, this is expected.</strong>
    </p>
    <p>
        We have already handled this initialization process within the code. You don't need to do anything.
    </p>
    <p>
        <strong>Simply wait for this cell to finish running, and then proceed to the next cell.</strong>
    </p>
</div>

In [ ]:
#@title 🧠 2. Global Initialization (Load All Models & Data)
print("⏳ Initializing and warming up the RDKit environment...")
try:
    from rdkit import Chem
    # If the first import succeeds, great.
except ImportError as e:
    # The first import may fail in a freshly restarted Colab runtime.
    # This is a known issue. We capture the error and retry,
    # as the failed attempt often prepares the environment for the next one.
    print(f"Caught an expected initialization error: {e}")
    print("Retrying import...")
    from rdkit import Chem

print("✅ RDKit environment is ready!")
print("💡💡💡 This is a normal phenomenon, please don't be surprised, wait for the program to finish running and run the next cell.!")

import sys
import os
import gzip
import torch
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from rdkit.Chem import Draw, MolFromSmiles
from google.colab import files
import pandas as pd
import jcamp
import numpy as np
from PIL import Image
import io

# --- 1. Set up project paths ---
print("--- Setting up project paths for module imports ---")
GDRIVE_REPO_PATH = "/content/drive/MyDrive/colab_repos/CSU-IR"
PS_CLASSIFIER_MODULE_PATH = os.path.join(GDRIVE_REPO_PATH, 'PS-Classifier')
if PS_CLASSIFIER_MODULE_PATH not in sys.path: sys.path.insert(0, PS_CLASSIFIER_MODULE_PATH)
print("✅ Project paths set.")

# --- 2. Import custom modules ---
print("\n--- Importing custom modules ---")
from model.IR_encoder import IRModel
from model.SMILES_encoder import SmilesModel
from model.Classifier_model import classifymodel
from data_process.ir_process import preprocess_spectra_higer_500, preprocess_spectra_lower_500
from test_and_infer.infer_SMILES_Classifier import normalize_smiles
print("✅ Custom modules imported.")

# --- 3. Define all file paths ---
print("\n--- Defining all file paths ---")
TOKENIZER_PATH = os.path.join(PS_CLASSIFIER_MODULE_PATH, "model", "tokenizer-smiles-roberta-1e_new")
PS_SMILES_ENCODER_PATH = os.path.join(DEST_PS_CLASSIFIER_WEIGHTS_PATH, "best_smiles_model_650-4000.pth")
PS_SMILES_CLASSIFIER_PATH = os.path.join(DEST_PS_CLASSIFIER_WEIGHTS_PATH, "best_Smiles_Classifier_model.pth")
TEST_DATA_PATH = os.path.join(PS_CLASSIFIER_MODULE_PATH, "data", "test_data")
print("✅ File paths defined.")

# --- 4. Initialize all models and load weights ---
print("\n--- Initializing and loading all models ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


# Classifier Models
# Note: The encoders for classifiers are separate instances

smiles_classifier_encoder = SmilesModel(roberta_model_path=None, roberta_tokenizer_path=TOKENIZER_PATH, feature_dim=768)
smiles_classifier_model = classifymodel(dim=1024, num_classes=2)


smiles_classifier_encoder.load_state_dict(torch.load(PS_SMILES_ENCODER_PATH, map_location=device))
smiles_classifier_model.load_state_dict(torch.load(PS_SMILES_CLASSIFIER_PATH, map_location=device))


smiles_classifier_encoder.to(device).eval()
smiles_classifier_model.to(device).eval()
print("✅ Classifier models loaded.")


print("\n\n🎉🎉🎉 Global initialization complete! System is ready! 🎉🎉🎉")

### **Step 3: Performance**

In [ ]:
#@title ⚛️ Task : One-Click SMILES Analysis
#@markdown Choose a test mode. You can see the results on our pre-loaded test set or enter a single SMILES string for instant classification.
#@markdown ---
test_mode_smiles = "See batch test results on our test set" #@param ["See batch test results on our test set", "See NPS test results", "Enter a SMILES string to instantly classify"]
smiles_to_predict = "CCCO" #@param {type:"string"}

# --- Define a new dataset class for SMILES Classification ---
class SmilesDataset(Dataset):
    def __init__(self, smiles, labels): self.smiles, self.labels = smiles, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.smiles[i], self.labels[i]

class NPSSmilesDataset(Dataset):
    def __init__(self, smiles): self.smiles = smiles
    def __len__(self): return len(self.smiles)
    def __getitem__(self, i): return self.smiles[i]

def load_smiles_list(path):
    with open(path, 'r', encoding='utf-8') as f: return [line.strip() for line in f if line.strip()]

if test_mode_smiles == "See batch test results on our test set":
    print("--- Running batch test for SMILES Classifier ---")
    # Load data
    test_smiles_list = load_smiles_list(os.path.join(TEST_DATA_PATH, "SMILES_Classifier", "test_smiles.txt"))
    test_label = torch.load(os.path.join(TEST_DATA_PATH, "SMILES_Classifier", "test_labels.pt"))
    test_loader = DataLoader(SmilesDataset(test_smiles_list, test_label), batch_size=208, shuffle=False)

    # Run inference
    all_preds, all_labels = [], []
    with torch.no_grad():
        # Use the correct classifier encoder model
        tokenizer = smiles_classifier_encoder.smiles_tokenizer
        for smiles_batch, label_batch in tqdm(test_loader, desc="SMILES Batch Test"):
            encoded = [tokenizer.encode_plus(s, max_length=300, padding='max_length', truncation=True, return_tensors='pt') for s in smiles_batch]
            input_ids = torch.cat([item['input_ids'] for item in encoded], dim=0).to(device)
            attention_mask = torch.cat([item['attention_mask'] for item in encoded], dim=0).to(device)
            lengths = attention_mask.sum(dim=1)

            smiles_features = smiles_classifier_encoder.encode((input_ids, attention_mask), lengths)
            logits = smiles_classifier_model(smiles_features)

            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(label_batch.cpu().numpy())

    # Calculate and print confusion matrix
    label_0_pred_0 = sum(1 for p, l in zip(all_preds, all_labels) if l == 0 and p == 0)
    label_0_pred_1 = sum(1 for p, l in zip(all_preds, all_labels) if l == 0 and p == 1)
    label_1_pred_1 = sum(1 for p, l in zip(all_preds, all_labels) if l == 1 and p == 1)
    label_1_pred_0 = sum(1 for p, l in zip(all_preds, all_labels) if l == 1 and p == 0)
    print(f"\n--- Batch Test Results ---")
    print(f"Number of non-PS predicted as non-PS: {label_0_pred_0}")
    print(f"Number of non-PS predicted as PS: {label_0_pred_1}")
    print(f"Number of PS predicted as PS: {label_1_pred_1}")
    print(f"Number of PS predicted as non-PS: {label_1_pred_0}")

elif test_mode_smiles == "See NPS test results":
    print("--- Running  test results for NPS ---")
    # Load data
    test_smiles_list = load_smiles_list(os.path.join(TEST_DATA_PATH, "SMILES_Classifier", "NPS_smiles.txt"))
    test_loader = DataLoader(NPSSmilesDataset(test_smiles_list), batch_size=208, shuffle=False)

    # Run inference
    all_preds = []
    with torch.no_grad():
        # Use the correct classifier encoder model
        tokenizer = smiles_classifier_encoder.smiles_tokenizer
        for smiles_batch in tqdm(test_loader, desc="SMILES Batch Test"):
            encoded = [tokenizer.encode_plus(s, max_length=300, padding='max_length', truncation=True, return_tensors='pt') for s in smiles_batch]
            input_ids = torch.cat([item['input_ids'] for item in encoded], dim=0).to(device)
            attention_mask = torch.cat([item['attention_mask'] for item in encoded], dim=0).to(device)
            lengths = attention_mask.sum(dim=1)

            smiles_features = smiles_classifier_encoder.encode((input_ids, attention_mask), lengths)
            logits = smiles_classifier_model(smiles_features)

            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())

    print("\n--- NPS Test Results ---")
    for i, pred in enumerate(all_preds):
        category = "Psychoactive Substance" if pred == 1 else "Non-Psychoactive Substance"
        print(f"Sample {i+1}: Predicted as -> {category}")


else: # Single SMILES string mode
    print(f"\n--- Predicting for SMILES: '{smiles_to_predict}' ---")

    # Define the prediction function specifically for this cell
    def predict_smiles_classification(smiles):
        if not smiles or not smiles.strip():
            return "⚠️ Error: Please enter a valid SMILES string."

        # Use the normalize_smiles function we defined in the global init cell
        normalized = normalize_smiles(smiles.strip())
        print(f"Normalized SMILES: {normalized}")

        # Use the correct classifier models
        tokenizer = smiles_classifier_encoder.smiles_tokenizer
        encoded = tokenizer.encode_plus(text=normalized, max_length=300, padding='max_length', truncation=True, return_tensors='pt')

        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        lengths = attention_mask.sum(dim=1)

        with torch.no_grad():
            smiles_features = smiles_classifier_encoder.encode((input_ids, attention_mask), lengths)
            logits = smiles_classifier_model(smiles_features)

        pred = torch.argmax(logits, dim=1).item()
        return "✅ Result: Psychoactive Substance" if pred == 1 else "✅ Result: Non-Psychoactive Substance"

    # Run prediction based on user input from the form
    result = predict_smiles_classification(smiles_to_predict)
    print(f"\n{result}")